# Análisis Geoespacial IPM - Zona de Intervención: Roosevelt

Este notebook prioriza la visualización cartográfica de las 5 variables más relevantes del Índice de Pobreza Multidimensional (IPM) en el corredor Roosevelt (Buffer 100m).

### Contexto y Cifras Generales (Corte Mayo 2026)

Para la interpretación de los resultados, se deben considerar dos lógicas opuestas:
1. **Índice de Condición Social (ICS):** Es un indicador directo; un valor del **100% representa el escenario óptimo** (bienestar y acceso pleno).
2. **Índice de Pobreza Multidimensional (IPM):** Es un indicador inverso; un valor del **100% representa el escenario crítico** (pobreza absoluta).

A nivel distrital, Cali presenta una marcada brecha territorial. La **severidad de la pobreza (IPM promedio en manzanas con incidencia > 0)** es del **15.28% en el área urbana**, mientras que en el **área rural se dispara al 38.33%**, impulsada principalmente por carencias en infraestructura básica en los corregimientos.

Este análisis se enfoca en el corredor Roosevelt para identificar las privaciones específicas que afectan a este territorio de intervención.

In [ ]:
# 1. Instalación de dependencias (solo en Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas matplotlib seaborn openpyxl -q

In [ ]:
# 2. Importar librerías
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from matplotlib.patheffects import withStroke
from matplotlib.colors import BoundaryNorm
import matplotlib.patches as mpatches

sns.set_style('white')
print('Librerías listas')

In [ ]:
# 3. Definir rutas con detección de entorno
REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !cd {REPO_DIR} && git pull
    BASE_DIR = REPO_DIR
else:
    if os.path.exists('indice_Pobreza'): BASE_DIR = '.'
    elif os.path.exists('../indice_Pobreza'): BASE_DIR = '..'
    elif os.path.exists('../../indice_Pobreza'): BASE_DIR = '../..'
    else: BASE_DIR = '.'

DATA_BASE = os.path.join(BASE_DIR, 'indice_Pobreza/data')

# Capas
PATH_IPM_VARS_FULL = os.path.join(DATA_BASE, 'geojson_ipm/Mzn_ipm_variables.geojson')
PATH_MANZANAS_FONDO = os.path.join(DATA_BASE, 'geojson_Manzanas_catastrales/geojson_Manzanas_catastrales.geojson')
PATH_AREA_ESTUDIO = os.path.join(DATA_BASE, 'geojson_poligonos_territorio_ITT/poligono_Roosevelt_Buffer_100.geojson')

# Cargar datos
gdf_full = gpd.read_file(PATH_IPM_VARS_FULL)
gdf_fondo = gpd.read_file(PATH_MANZANAS_FONDO)
gdf_area = gpd.read_file(PATH_AREA_ESTUDIO)

# Unificar CRS a WGS84
gdf_full = gdf_full.to_crs('EPSG:4326')
gdf_fondo = gdf_fondo.to_crs('EPSG:4326')
gdf_area = gdf_area.to_crs('EPSG:4326')

# Filtrado espacial preciso (Centroide dentro del Buffer)
gdf_p = gdf_full.to_crs("EPSG:3115")
area_p = gdf_area.to_crs("EPSG:3115")
idx = gpd.sjoin(gdf_p.copy().assign(geometry=gdf_p.centroid), area_p[['geometry']], how='inner', predicate='within').index
gdf = gdf_full.loc[idx].copy()

print(f'Manzanas detectadas en Roosevelt: {len(gdf)}')

In [ ]:
# 4. Diccionario Estandarizado y Top 5
COLS_MAP = {
    'ANALF_': 'Analfabetismo', 'BAJO_': 'Bajo logro educativo', 
    'INFANCIA_': 'Barreras primera infancia', 'INASIS_': 'Inasistencia escolar', 
    'REZAGO_': 'Rezago escolar', 'TRAB_INFAN': 'Trabajo infantil', 
    'DEPEN_': 'Dependencia económica', 'INFOR_': 'Informalidad', 
    'SALUD_': 'Barreras de salud', 'ASEGU_': 'Sin aseguramiento en salud', 
    'HACI_': 'Hacinamiento crítico', 'PARED_': 'Paredes precarias', 
    'EXCRE_': 'Eliminación inadecuada de excretas', 'PISOS_': 'Pisos precarios', 
    'AGUA_': 'Sin acceso a agua mejorada'
}

top_5 = gdf[list(COLS_MAP.keys())].mean().sort_values(ascending=False).head(5).index.tolist()
print('Top 5 variables críticas en Roosevelt:')
for v in top_5: print(f'- {COLS_MAP[v]}')

In [ ]:
# 5. Función de mapeo avanzada: Estándar Geo-informática Cali (Mayo 2026)
def plot_ipm_variable_advanced(gdf_zone, gdf_back, gdf_poly, column, title):
    """
    Genera cartografía profesional siguiendo las reglas oficiales:
    - Efecto Atlas: Fondo completo sin bordes blancos.
    - Layout: Zona centrada a la izquierda, espacio para leyenda a la derecha (+75% X).
    - Contraste: Etiquetas dinámicas con stroke (B/N).
    """
    # Configuración de lienzo sin bordes
    fig, ax = plt.subplots(1, 1, figsize=(22, 12), facecolor='white')
    plt.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.01)
    
    # --- 1. Cálculo de Zoom Dinámico y Expansión de Lienzo ---
    bounds = gdf_poly.total_bounds # [minx, miny, maxx, maxy]
    width = bounds[2] - bounds[0]
    height = bounds[3] - bounds[1]
    
    # Aplicar regla de expansión (+75% a la derecha para leyenda y +10% margen general)
    view_minx = bounds[0] - (width * 0.1)
    view_maxx = bounds[2] + (width * 0.85) # Espacio para leyenda
    view_miny = bounds[1] - (height * 0.1)
    view_maxy = bounds[3] + (height * 0.1)
    
    # --- 2. Rangos Dinámicos (5 niveles + 0) ---
    vals = gdf_zone[column].dropna()
    if not vals.empty and vals.max() > 0:
        breaks = sorted(list(set([0, 0.1] + list(np.linspace(vals[vals>0].min() if not vals[vals>0].empty else 1, vals.max(), 4)))))
    else:
        breaks = [0, 1, 2, 3, 4, 5]
    
    n_bins = len(breaks) - 1
    cmap = plt.get_cmap('viridis', n_bins)
    norm = BoundaryNorm(breaks, cmap.N)
    
    # --- 3. Ploteo de Capas (Orden Z-Order) ---
    # Capa 1: Fondo de Manzanas Catastrales (Cubre TODO el lienzo)
    gdf_back.plot(ax=ax, facecolor='#F5F5F2', edgecolor='#DCDCDC', linewidth=0.3, alpha=1.0, zorder=1)
    
    # Capa 2: Polígono de la Zona (Referencia ITT)
    gdf_poly.plot(ax=ax, facecolor='none', edgecolor='#C0392B', linewidth=2.5, linestyle='--', alpha=0.8, zorder=2)
    
    # Capa 3: Manzanas con datos IPM (Coropleta)
    gdf_zone.plot(column=column, cmap=cmap, norm=norm, edgecolor='white', linewidth=0.8, ax=ax, zorder=3, alpha=0.95)
    
    # --- 4. Etiquetas de Contraste Inteligente ---
    c = gdf_zone.to_crs("EPSG:3115").geometry.centroid.to_crs(gdf_zone.crs)
    
    for x, y, label in zip(c.x, c.y, gdf_zone[column]):
        is_high = label > (vals.max() * 0.5) if not vals.empty else False
        txt_color = 'white' if is_high else 'black'
        stk_color = 'black' if is_high else 'white'
        
        ax.annotate(f'{label:.1f}%', xy=(x, y), ha='center', va='center', 
                    fontsize=10, fontweight='black', color=txt_color,
                    path_effects=[withStroke(linewidth=2, foreground=stk_color, alpha=0.9)],
                    zorder=5)
    
    # --- 5. Leyenda Integrada (DENTRO de los ejes a la derecha) ---
    legend_patches = []
    for i in range(n_bins):
        lbl = f"{breaks[i]:.1f}% - {breaks[i+1]:.1f}%"
        legend_patches.append(mpatches.Patch(facecolor=cmap(i), edgecolor='gray', label=lbl))
    
    legend_patches.append(mpatches.Patch(facecolor='none', edgecolor='#C0392B', 
                                         linewidth=2, linestyle='--', label='Buffer Roosevelt (100m)'))
    
    ax.legend(handles=legend_patches, 
              title=f"Privación: {title}\n(% Hogares)", 
              loc='center left', bbox_to_anchor=(0.70, 0.5), 
              fontsize=10, title_fontsize=12, framealpha=0.95, edgecolor='#CCCCCC')
    
    ax.set_title(f"Distribución de {title.upper()} - Zona Roosevelt", 
                 fontsize=18, fontweight='bold', color='#2C3E50', pad=30)
    
    ax.set_xlim(view_minx, view_maxx)
    ax.set_ylim(view_miny, view_maxy)
    ax.set_axis_off()
    plt.show()

In [ ]:
# 6. Ejecución de la Cartografía Crítica (Top 5)
print("Generando mapas bajo estándar oficial 'Efecto Atlas'...")
for var in top_5:
    plot_ipm_variable_advanced(gdf, gdf_fondo, gdf_area, var, COLS_MAP[var])

# Exportar GeoJSON filtrado
gdf.to_file('Mzn_ipm_variables_Roosevelt.geojson', driver='GeoJSON')

## 7. Validación de Estándares Técnicos
Este notebook cumple con las directrices de geo-informática de Santiago de Cali (Mayo 2026):
1. **Accesibilidad:** Uso de la paleta **Viridis** para IPM.
2. **Efecto Atlas:** Manzanas de contexto y fondo completo (#F5F5F2).
3. **Contraste:** Etiquetas inteligentes (Blanco/Negro) con stroke.
4. **Layout:** Zona centrada a la izquierda con leyenda interna a la derecha.
5. **Precisión:** Datos ajustados a **1 decimal**.

In [ ]:
# 8. Descargar GeoJSON (Colab)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('Mzn_ipm_variables_Roosevelt.geojson')
    print('Descarga iniciada: Mzn_ipm_variables_Roosevelt.geojson')
else:
    print(f'Archivo disponible localmente.')